<a href="https://colab.research.google.com/github/poojaullegaddi/AI-Chatbot-NLP/blob/main/tiny_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [50]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-11 17:19:48--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.3’

input.txt.3         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-04-11 17:19:48 (31.3 MB/s) - ‘input.txt.3’ saved [1115394/1115394]



In [51]:
text = open('input.txt', 'r').read()

In [52]:
print(len(text))
print(text[:200])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [53]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [54]:
data = torch.tensor(encode(text), dtype=torch.long)

In [55]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [56]:
block_size = 32
batch_size = 4
n_embd = 128    # embedding size
n_head = 4     # number of attention heads
n_layer = 2    # number of transformer blocks


In [57]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

In [58]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        v = self.value(x)
        out = wei @ v
        return out

In [59]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [60]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

In [61]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [62]:
class GPTLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb

        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]

            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)

            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)

        return idx

In [63]:
model = GPTLanguageModel(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for steps in range(15000):

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if steps % 500 == 0:
        print("Step:", steps, "Loss:", loss.item())

Step: 0 Loss: 4.273197174072266
Step: 500 Loss: 2.24211049079895
Step: 1000 Loss: 2.303654432296753
Step: 1500 Loss: 2.370380163192749
Step: 2000 Loss: 1.8096400499343872
Step: 2500 Loss: 1.8701997995376587
Step: 3000 Loss: 1.6740349531173706
Step: 3500 Loss: 1.919091820716858
Step: 4000 Loss: 1.7227052450180054
Step: 4500 Loss: 1.9132132530212402
Step: 5000 Loss: 1.6519449949264526
Step: 5500 Loss: 1.8653596639633179
Step: 6000 Loss: 1.8032240867614746
Step: 6500 Loss: 1.7275911569595337
Step: 7000 Loss: 1.6286007165908813
Step: 7500 Loss: 1.8109489679336548
Step: 8000 Loss: 1.7558342218399048
Step: 8500 Loss: 1.5096755027770996
Step: 9000 Loss: 1.479844570159912
Step: 9500 Loss: 1.7902896404266357
Step: 10000 Loss: 1.696160078048706
Step: 10500 Loss: 1.7120689153671265
Step: 11000 Loss: 1.3334993124008179
Step: 11500 Loss: 1.67037034034729
Step: 12000 Loss: 1.7913204431533813
Step: 12500 Loss: 1.7317390441894531
Step: 13000 Loss: 1.7728451490402222
Step: 13500 Loss: 1.636977672576904

In [64]:
context = torch.zeros((1,1), dtype=torch.long)

generated = model.generate(context, max_new_tokens=300)

print(decode(generated[0].tolist()))



CESTRUCHIO:
In you courter hear.

QUEEN:
Your he the days, but be stagues, if sorrows
Which is forceived, Take may how or, is
Nother from the happ ear the brother's king hath singin Boge, royal thou warthings I my woady.

BUCKINGHAM:
'Tworm your gentlemerr'st bys he
With was?

First Citizen:
Yourin
